In [1]:
# ============================================================
# 1. Configuración inicial del proyecto
# SQL Business Analysis Project
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

# Para reproducibilidad
np.random.seed(42)

# Carpeta principal del proyecto
project_root = Path("/content/sql-business-analysis-project")

# Carpetas del proyecto
folders = [
    project_root,
    project_root / "data",
    project_root / "data" / "raw",
    project_root / "data" / "processed",
    project_root / "sql",
    project_root / "reports",
    project_root / "dashboard"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Proyecto inicializado correctamente en:")
print(project_root)

print("\nCarpetas creadas:")
for folder in folders:
    print("-", folder)

Proyecto inicializado correctamente en:
/content/sql-business-analysis-project

Carpetas creadas:
- /content/sql-business-analysis-project
- /content/sql-business-analysis-project/data
- /content/sql-business-analysis-project/data/raw
- /content/sql-business-analysis-project/data/processed
- /content/sql-business-analysis-project/sql
- /content/sql-business-analysis-project/reports
- /content/sql-business-analysis-project/dashboard


In [2]:
# ============================================================
# 2. Creación de datasets simulados de negocio
# SQL Business Analysis Project
# ============================================================

# Número de registros
n_customers = 8000
n_orders = 25000
n_transactions = 25000
n_products = 120
n_support_tickets = 6000

# Fechas
start_date = pd.to_datetime("2024-01-01")
end_date = pd.to_datetime("2024-12-31")

# ----------------------------
# Tabla customers
# ----------------------------

countries = ["Spain", "France", "Germany", "Italy", "Portugal", "Netherlands", "United Kingdom"]
segments = ["low_value", "mid_value", "high_value"]
acquisition_channels = ["organic", "paid_search", "social_media", "referral", "email"]

customers = pd.DataFrame({
    "customer_id": np.arange(100000, 100000 + n_customers),
    "country": np.random.choice(countries, n_customers),
    "customer_segment": np.random.choice(segments, n_customers, p=[0.45, 0.40, 0.15]),
    "acquisition_channel": np.random.choice(acquisition_channels, n_customers),
    "signup_date": start_date + pd.to_timedelta(np.random.randint(0, 365, n_customers), unit="D")
})

customers["signup_month"] = customers["signup_date"].dt.to_period("M").astype(str)

# ----------------------------
# Tabla products
# ----------------------------

product_categories = ["electronics", "home", "fashion", "sports", "beauty", "books", "toys"]

products = pd.DataFrame({
    "product_id": np.arange(2000, 2000 + n_products),
    "category": np.random.choice(product_categories, n_products),
    "unit_cost": np.round(np.random.uniform(5, 120, n_products), 2),
    "unit_price": np.round(np.random.uniform(12, 250, n_products), 2)
})

products["margin_per_unit"] = (products["unit_price"] - products["unit_cost"]).round(2)

# ----------------------------
# Tabla orders
# ----------------------------

orders = pd.DataFrame({
    "order_id": np.arange(500000, 500000 + n_orders),
    "customer_id": np.random.choice(customers["customer_id"], n_orders),
    "product_id": np.random.choice(products["product_id"], n_orders),
    "order_date": start_date + pd.to_timedelta(np.random.randint(0, 365, n_orders), unit="D"),
    "quantity": np.random.randint(1, 6, n_orders),
    "discount_pct": np.random.choice([0, 5, 10, 15, 20], n_orders, p=[0.50, 0.20, 0.15, 0.10, 0.05])
})

orders = orders.merge(products[["product_id", "unit_price", "unit_cost"]], on="product_id", how="left")

orders["gross_revenue"] = (orders["quantity"] * orders["unit_price"]).round(2)
orders["discount_amount"] = (orders["gross_revenue"] * orders["discount_pct"] / 100).round(2)
orders["net_revenue"] = (orders["gross_revenue"] - orders["discount_amount"]).round(2)
orders["total_cost"] = (orders["quantity"] * orders["unit_cost"]).round(2)
orders["gross_profit"] = (orders["net_revenue"] - orders["total_cost"]).round(2)
orders["order_month"] = orders["order_date"].dt.to_period("M").astype(str)

# ----------------------------
# Tabla transactions
# ----------------------------

payment_methods = ["card", "paypal", "bank_transfer", "apple_pay", "google_pay"]
transaction_statuses = ["approved", "declined", "refunded", "chargeback"]

transactions = pd.DataFrame({
    "transaction_id": np.arange(900000, 900000 + n_transactions),
    "order_id": orders["order_id"],
    "payment_method": np.random.choice(payment_methods, n_transactions),
    "transaction_status": np.random.choice(
        transaction_statuses,
        n_transactions,
        p=[0.88, 0.07, 0.03, 0.02]
    )
})

transactions = transactions.merge(
    orders[["order_id", "customer_id", "order_date", "net_revenue"]],
    on="order_id",
    how="left"
)

transactions["transaction_month"] = transactions["order_date"].dt.to_period("M").astype(str)

# Simulamos riesgo de fraude
transactions["fraud_flag"] = np.where(
    (transactions["transaction_status"] == "chargeback") |
    ((transactions["net_revenue"] > transactions["net_revenue"].quantile(0.95)) &
     (transactions["payment_method"].isin(["card", "paypal"]))),
    1,
    0
)

# ----------------------------
# Tabla support_tickets
# ----------------------------

ticket_categories = ["billing", "delivery", "technical_issue", "refund", "account_access"]
ticket_priorities = ["low", "medium", "high", "urgent"]

support_tickets = pd.DataFrame({
    "ticket_id": np.arange(700000, 700000 + n_support_tickets),
    "customer_id": np.random.choice(customers["customer_id"], n_support_tickets),
    "created_at": start_date + pd.to_timedelta(np.random.randint(0, 365, n_support_tickets), unit="D"),
    "ticket_category": np.random.choice(ticket_categories, n_support_tickets),
    "priority": np.random.choice(ticket_priorities, n_support_tickets, p=[0.40, 0.40, 0.15, 0.05]),
    "resolution_hours": np.round(np.random.gamma(shape=3, scale=8, size=n_support_tickets), 2)
})

support_tickets["ticket_month"] = support_tickets["created_at"].dt.to_period("M").astype(str)
support_tickets["sla_met"] = np.where(support_tickets["resolution_hours"] <= 48, 1, 0)

# ----------------------------
# Guardar CSVs raw
# ----------------------------

raw_path = project_root / "data" / "raw"

customers.to_csv(raw_path / "customers.csv", index=False)
products.to_csv(raw_path / "products.csv", index=False)
orders.to_csv(raw_path / "orders.csv", index=False)
transactions.to_csv(raw_path / "transactions.csv", index=False)
support_tickets.to_csv(raw_path / "support_tickets.csv", index=False)

print("Datasets creados y guardados correctamente en:")
print(raw_path)

print("\nTablas creadas:")
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("transactions:", transactions.shape)
print("support_tickets:", support_tickets.shape)

display(customers.head())
display(orders.head())
display(transactions.head())

Datasets creados y guardados correctamente en:
/content/sql-business-analysis-project/data/raw

Tablas creadas:
customers: (8000, 6)
products: (120, 5)
orders: (25000, 14)
transactions: (25000, 9)
support_tickets: (6000, 8)


,customer_id,country,customer_segment,acquisition_channel,signup_date,signup_month
0,100000,United Kingdom,low_value,email,2024-08-26,2024-08
1,100001,Italy,high_value,email,2024-10-29,2024-10
2,100002,Portugal,low_value,organic,2024-06-13,2024-06
3,100003,United Kingdom,low_value,social_media,2024-08-13,2024-08
4,100004,Germany,high_value,social_media,2024-06-24,2024-06


,order_id,customer_id,product_id,order_date,quantity,discount_pct,unit_price,unit_cost,gross_revenue,discount_amount,net_revenue,total_cost,gross_profit,order_month
0,500000,103197,2016,2024-02-24,3,10,239.72,117.33,719.16,71.92,647.24,351.99,295.25,2024-02
1,500001,101182,2071,2024-10-20,1,20,26.98,100.97,26.98,5.40,21.58,100.97,-79.39,2024-10
2,500002,105790,2041,2024-09-16,4,0,59.98,40.43,239.92,0.00,239.92,161.72,78.20,2024-09
3,500003,100794,2016,2024-01-15,4,0,239.72,117.33,958.88,0.00,958.88,469.32,489.56,2024-01
4,500004,101624,2047,2024-05-28,4,5,64.30,116.03,257.20,12.86,244.34,464.12,-219.78,2024-05


,transaction_id,order_id,payment_method,transaction_status,customer_id,order_date,net_revenue,transaction_month,fraud_flag
0,900000,500000,card,approved,103197,2024-02-24,647.24,2024-02,0
1,900001,500001,apple_pay,approved,101182,2024-10-20,21.58,2024-10,0
2,900002,500002,paypal,approved,105790,2024-09-16,239.92,2024-09,0
3,900003,500003,paypal,approved,100794,2024-01-15,958.88,2024-01,1
4,900004,500004,card,approved,101624,2024-05-28,244.34,2024-05,0


In [3]:
# ============================================================
# 3. Registro de tablas en DuckDB y validación inicial
# SQL Business Analysis Project
# ============================================================

# Crear conexión DuckDB
conn = duckdb.connect()

# Registrar dataframes como tablas SQL
conn.register("customers", customers)
conn.register("products", products)
conn.register("orders", orders)
conn.register("transactions", transactions)
conn.register("support_tickets", support_tickets)

# Validaciones básicas con SQL
validation_query = """
SELECT 'customers' AS table_name, COUNT(*) AS rows_count FROM customers
UNION ALL
SELECT 'products' AS table_name, COUNT(*) AS rows_count FROM products
UNION ALL
SELECT 'orders' AS table_name, COUNT(*) AS rows_count FROM orders
UNION ALL
SELECT 'transactions' AS table_name, COUNT(*) AS rows_count FROM transactions
UNION ALL
SELECT 'support_tickets' AS table_name, COUNT(*) AS rows_count FROM support_tickets;
"""

validation_summary = conn.execute(validation_query).df()

# Validación de claves principales
primary_key_checks = pd.DataFrame({
    "table_name": [
        "customers",
        "products",
        "orders",
        "transactions",
        "support_tickets"
    ],
    "id_column": [
        "customer_id",
        "product_id",
        "order_id",
        "transaction_id",
        "ticket_id"
    ],
    "duplicated_ids": [
        customers["customer_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        transactions["transaction_id"].duplicated().sum(),
        support_tickets["ticket_id"].duplicated().sum()
    ],
    "missing_values": [
        customers.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        transactions.isna().sum().sum(),
        support_tickets.isna().sum().sum()
    ]
})

# Guardamos validaciones
processed_path = project_root / "data" / "processed"

validation_summary.to_csv(processed_path / "validation_summary.csv", index=False)
primary_key_checks.to_csv(processed_path / "primary_key_checks.csv", index=False)

print("Resumen de filas por tabla:")
display(validation_summary)

print("Validación de claves principales:")
display(primary_key_checks)

print("Validaciones guardadas correctamente en:")
print(processed_path)

Resumen de filas por tabla:


,table_name,rows_count
0,customers,8000
1,products,120
2,orders,25000
3,transactions,25000
4,support_tickets,6000


Validación de claves principales:


,table_name,id_column,duplicated_ids,missing_values
0,customers,customer_id,0,0
1,products,product_id,0,0
2,orders,order_id,0,0
3,transactions,transaction_id,0,0
4,support_tickets,ticket_id,0,0


Validaciones guardadas correctamente en:
/content/sql-business-analysis-project/data/processed


In [4]:
# ============================================================
# 4. KPIs generales de negocio con SQL
# SQL Business Analysis Project
# ============================================================

query_01_business_kpis = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS active_customers,
    ROUND(SUM(o.net_revenue), 2) AS total_net_revenue,
    ROUND(SUM(o.gross_profit), 2) AS total_gross_profit,
    ROUND(AVG(o.net_revenue), 2) AS avg_order_value,
    ROUND(AVG(o.gross_profit), 2) AS avg_profit_per_order,
    ROUND(SUM(o.gross_profit) / NULLIF(SUM(o.net_revenue), 0) * 100, 2) AS gross_margin_pct
FROM orders o;
"""

business_kpis = conn.execute(query_01_business_kpis).df()

# Guardar consulta SQL
sql_path = project_root / "sql"
with open(sql_path / "01_business_kpis.sql", "w", encoding="utf-8") as f:
    f.write(query_01_business_kpis.strip())

# Guardar resultado
business_kpis.to_csv(project_root / "data" / "processed" / "01_business_kpis.csv", index=False)

print("KPIs generales de negocio:")
display(business_kpis)

print("Consulta SQL y resultado guardados correctamente.")

KPIs generales de negocio:


,total_orders,active_customers,total_net_revenue,total_gross_profit,avg_order_value,avg_profit_per_order,gross_margin_pct
0,25000,7627,8925618.18,4272047.86,357.02,170.88,47.86


Consulta SQL y resultado guardados correctamente.


In [5]:
# ============================================================
# 5. Análisis mensual de revenue y margen con SQL
# SQL Business Analysis Project
# ============================================================

query_02_monthly_revenue = """
SELECT
    order_month,
    COUNT(DISTINCT order_id) AS orders,
    COUNT(DISTINCT customer_id) AS active_customers,
    ROUND(SUM(net_revenue), 2) AS total_net_revenue,
    ROUND(SUM(gross_profit), 2) AS total_gross_profit,
    ROUND(AVG(net_revenue), 2) AS avg_order_value,
    ROUND(SUM(gross_profit) / NULLIF(SUM(net_revenue), 0) * 100, 2) AS gross_margin_pct
FROM orders
GROUP BY order_month
ORDER BY order_month;
"""

monthly_revenue = conn.execute(query_02_monthly_revenue).df()

# Guardar consulta SQL
with open(sql_path / "02_monthly_revenue.sql", "w", encoding="utf-8") as f:
    f.write(query_02_monthly_revenue.strip())

# Guardar resultado
monthly_revenue.to_csv(
    project_root / "data" / "processed" / "02_monthly_revenue.csv",
    index=False
)

print("Análisis mensual de revenue:")
display(monthly_revenue)

print("Consulta SQL y resultado guardados correctamente.")

Análisis mensual de revenue:


,order_month,orders,active_customers,total_net_revenue,total_gross_profit,avg_order_value,gross_margin_pct
0,2024-01,2139,1881,750710.56,346755.87,350.96,46.19
1,2024-02,1981,1736,707481.02,339149.83,357.13,47.94
2,2024-03,2105,1861,748820.11,362216.59,355.73,48.37
3,2024-04,2033,1802,729337.68,349241.35,358.75,47.88
4,2024-05,2128,1861,756074.78,359393.74,355.30,47.53
5,2024-06,2100,1849,745618.88,360689.93,355.06,48.37
6,2024-07,2125,1866,750368.87,355500.21,353.11,47.38
7,2024-08,2177,1911,763922.54,355112.03,350.91,46.49
8,2024-09,2042,1805,743662.97,368078.61,364.18,49.50
9,2024-10,2080,1834,738213.96,361912.11,354.91,49.03


Consulta SQL y resultado guardados correctamente.


In [6]:
# ============================================================
# 6. Análisis de clientes por país y segmento con SQL
# SQL Business Analysis Project
# ============================================================

query_03_customer_segments = """
SELECT
    c.country,
    c.customer_segment,
    COUNT(DISTINCT c.customer_id) AS customers,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(o.net_revenue), 2) AS total_net_revenue,
    ROUND(SUM(o.gross_profit), 2) AS total_gross_profit,
    ROUND(AVG(o.net_revenue), 2) AS avg_order_value,
    ROUND(SUM(o.gross_profit) / NULLIF(SUM(o.net_revenue), 0) * 100, 2) AS gross_margin_pct
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY
    c.country,
    c.customer_segment
ORDER BY
    total_net_revenue DESC;
"""

customer_segments = conn.execute(query_03_customer_segments).df()

# Guardar consulta SQL
with open(sql_path / "03_customer_segments.sql", "w", encoding="utf-8") as f:
    f.write(query_03_customer_segments.strip())

# Guardar resultado
customer_segments.to_csv(
    project_root / "data" / "processed" / "03_customer_segments.csv",
    index=False
)

print("Análisis de clientes por país y segmento:")
display(customer_segments)

print("Consulta SQL y resultado guardados correctamente.")

Análisis de clientes por país y segmento:


,country,customer_segment,customers,orders,total_net_revenue,total_gross_profit,avg_order_value,gross_margin_pct
0,Germany,low_value,527,1698,624305.98,307715.35,367.67,49.29
1,United Kingdom,low_value,536,1710,623691.85,298184.98,364.73,47.81
2,Spain,low_value,536,1718,603805.34,287289.12,351.46,47.58
3,Italy,low_value,543,1711,600205.44,280917.99,350.79,46.80
4,Portugal,low_value,530,1681,594601.00,284975.02,353.72,47.93
5,Netherlands,low_value,507,1563,557526.63,264108.45,356.70,47.37
6,France,low_value,488,1553,554377.34,260458.82,356.97,46.98
7,Portugal,mid_value,474,1445,521616.12,255270.10,360.98,48.94
8,Italy,mid_value,458,1435,512726.90,246925.86,357.30,48.16
9,France,mid_value,449,1406,502223.58,239386.44,357.20,47.67


Consulta SQL y resultado guardados correctamente.


In [7]:
# ============================================================
# 7. Análisis de productos y categorías con SQL
# SQL Business Analysis Project
# ============================================================

query_04_product_performance = """
SELECT
    p.category,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(o.quantity) AS units_sold,
    ROUND(SUM(o.net_revenue), 2) AS total_net_revenue,
    ROUND(SUM(o.gross_profit), 2) AS total_gross_profit,
    ROUND(AVG(o.net_revenue), 2) AS avg_order_value,
    ROUND(SUM(o.gross_profit) / NULLIF(SUM(o.net_revenue), 0) * 100, 2) AS gross_margin_pct
FROM orders o
LEFT JOIN products p
    ON o.product_id = p.product_id
GROUP BY
    p.category
ORDER BY
    total_net_revenue DESC;
"""

product_performance = conn.execute(query_04_product_performance).df()

# Guardar consulta SQL
with open(sql_path / "04_product_performance.sql", "w", encoding="utf-8") as f:
    f.write(query_04_product_performance.strip())

# Guardar resultado
product_performance.to_csv(
    project_root / "data" / "processed" / "04_product_performance.csv",
    index=False
)

print("Análisis de productos por categoría:")
display(product_performance)

print("Consulta SQL y resultado guardados correctamente.")

Análisis de productos por categoría:


,category,orders,units_sold,total_net_revenue,total_gross_profit,avg_order_value,gross_margin_pct
0,electronics,4811,14333.0,1929825.59,1069873.16,401.13,55.44
1,books,4552,13692.0,1731934.40,915778.71,380.48,52.88
2,toys,4428,13407.0,1494482.55,800055.98,337.51,53.53
3,home,4249,12625.0,1446576.26,576030.23,340.45,39.82
4,beauty,3563,10853.0,1331848.09,679983.90,373.80,51.06
5,fashion,2096,6215.0,631841.45,78385.90,301.45,12.41
6,sports,1301,3936.0,359109.84,151939.98,276.03,42.31


Consulta SQL y resultado guardados correctamente.


In [8]:
# ============================================================
# 8. Análisis de fraude, chargebacks y riesgo con SQL
# SQL Business Analysis Project
# ============================================================

query_05_fraud_risk = """
SELECT
    t.payment_method,
    COUNT(DISTINCT t.transaction_id) AS transactions,
    SUM(CASE WHEN t.transaction_status = 'approved' THEN 1 ELSE 0 END) AS approved_transactions,
    SUM(CASE WHEN t.transaction_status = 'declined' THEN 1 ELSE 0 END) AS declined_transactions,
    SUM(CASE WHEN t.transaction_status = 'refunded' THEN 1 ELSE 0 END) AS refunded_transactions,
    SUM(CASE WHEN t.transaction_status = 'chargeback' THEN 1 ELSE 0 END) AS chargebacks,
    SUM(t.fraud_flag) AS fraud_flagged_transactions,
    ROUND(SUM(t.net_revenue), 2) AS total_transaction_value,
    ROUND(AVG(t.net_revenue), 2) AS avg_transaction_value,
    ROUND(SUM(t.fraud_flag) * 100.0 / COUNT(DISTINCT t.transaction_id), 2) AS fraud_rate_pct,
    ROUND(
        SUM(CASE WHEN t.transaction_status = 'chargeback' THEN 1 ELSE 0 END) * 100.0
        / COUNT(DISTINCT t.transaction_id),
        2
    ) AS chargeback_rate_pct
FROM transactions t
GROUP BY
    t.payment_method
ORDER BY
    fraud_rate_pct DESC;
"""

fraud_risk = conn.execute(query_05_fraud_risk).df()

# Guardar consulta SQL
with open(sql_path / "05_fraud_risk.sql", "w", encoding="utf-8") as f:
    f.write(query_05_fraud_risk.strip())

# Guardar resultado
fraud_risk.to_csv(
    project_root / "data" / "processed" / "05_fraud_risk.csv",
    index=False
)

print("Análisis de fraude y riesgo por método de pago:")
display(fraud_risk)

print("Consulta SQL y resultado guardados correctamente.")

Análisis de fraude y riesgo por método de pago:


,payment_method,transactions,approved_transactions,declined_transactions,refunded_transactions,chargebacks,fraud_flagged_transactions,total_transaction_value,avg_transaction_value,fraud_rate_pct,chargeback_rate_pct
0,card,4993,4351.0,350.0,177.0,115.0,355.0,1808092.17,362.13,7.11,2.30
1,paypal,4980,4390.0,336.0,162.0,92.0,323.0,1765577.92,354.53,6.49,1.85
2,apple_pay,5047,4430.0,337.0,154.0,126.0,126.0,1817699.16,360.15,2.50,2.50
3,bank_transfer,5021,4402.0,361.0,154.0,104.0,104.0,1783103.96,355.13,2.07,2.07
4,google_pay,4959,4379.0,350.0,143.0,87.0,87.0,1751144.97,353.12,1.75,1.75


Consulta SQL y resultado guardados correctamente.


In [9]:
# ============================================================
# 9. Análisis de soporte, SLA y tickets con SQL
# SQL Business Analysis Project
# ============================================================

query_06_support_sla = """
SELECT
    ticket_category,
    priority,
    COUNT(DISTINCT ticket_id) AS tickets,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    SUM(sla_met) AS sla_met_tickets,
    ROUND(AVG(sla_met) * 100, 2) AS sla_met_pct
FROM support_tickets
GROUP BY
    ticket_category,
    priority
ORDER BY
    tickets DESC;
"""

support_sla = conn.execute(query_06_support_sla).df()

# Guardar consulta SQL
with open(sql_path / "06_support_sla.sql", "w", encoding="utf-8") as f:
    f.write(query_06_support_sla.strip())

# Guardar resultado
support_sla.to_csv(
    project_root / "data" / "processed" / "06_support_sla.csv",
    index=False
)

print("Análisis de soporte y SLA:")
display(support_sla)

print("Consulta SQL y resultado guardados correctamente.")

Análisis de soporte y SLA:


,ticket_category,priority,tickets,avg_resolution_hours,sla_met_tickets,sla_met_pct
0,account_access,low,531,22.97,499.0,93.97
1,billing,low,507,23.61,481.0,94.87
2,delivery,medium,506,24.24,475.0,93.87
3,refund,medium,491,23.93,458.0,93.28
4,account_access,medium,481,24.55,449.0,93.35
5,technical_issue,medium,481,23.57,460.0,95.63
6,technical_issue,low,472,22.70,450.0,95.34
7,billing,medium,461,24.04,428.0,92.84
8,delivery,low,455,24.37,433.0,95.16
9,refund,low,447,24.31,418.0,93.51


Consulta SQL y resultado guardados correctamente.


In [10]:
# ============================================================
# 10. Análisis combinado de clientes: revenue, fraude y soporte
# SQL Business Analysis Project
# ============================================================

query_07_customer_360 = """
SELECT
    c.customer_id,
    c.country,
    c.customer_segment,
    c.acquisition_channel,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(o.net_revenue), 2) AS total_net_revenue,
    ROUND(SUM(o.gross_profit), 2) AS total_gross_profit,
    COUNT(DISTINCT t.transaction_id) AS transactions,
    SUM(t.fraud_flag) AS fraud_flagged_transactions,
    COUNT(DISTINCT st.ticket_id) AS support_tickets,
    ROUND(AVG(st.resolution_hours), 2) AS avg_ticket_resolution_hours,
    ROUND(AVG(st.sla_met) * 100, 2) AS support_sla_met_pct
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
LEFT JOIN transactions t
    ON o.order_id = t.order_id
LEFT JOIN support_tickets st
    ON c.customer_id = st.customer_id
GROUP BY
    c.customer_id,
    c.country,
    c.customer_segment,
    c.acquisition_channel
ORDER BY
    total_net_revenue DESC;
"""

customer_360 = conn.execute(query_07_customer_360).df()

# Guardar consulta SQL
with open(sql_path / "07_customer_360.sql", "w", encoding="utf-8") as f:
    f.write(query_07_customer_360.strip())

# Guardar resultado
customer_360.to_csv(
    project_root / "data" / "processed" / "07_customer_360.csv",
    index=False
)

print("Vista cliente 360:")
display(customer_360.head(20))

print("Consulta SQL y resultado guardados correctamente.")

Vista cliente 360:


,customer_id,country,customer_segment,acquisition_channel,orders,total_net_revenue,total_gross_profit,transactions,fraud_flagged_transactions,support_tickets,avg_ticket_resolution_hours,support_sla_met_pct
0,100412,Netherlands,mid_value,organic,6,12030.93,7347.96,6,3.0,3,15.23,100.00
1,103674,Portugal,mid_value,referral,8,11498.01,7682.52,8,6.0,3,20.40,100.00
2,103349,France,high_value,email,5,10559.40,6726.00,5,0.0,3,50.64,33.33
3,102934,Italy,low_value,paid_search,10,10329.51,5787.54,10,3.0,3,45.24,66.67
4,105730,France,low_value,organic,8,10140.04,5832.68,8,4.0,4,16.53,100.00
5,105998,Portugal,high_value,paid_search,7,10052.60,3216.48,7,0.0,4,22.01,100.00
6,104206,Portugal,low_value,organic,6,9855.69,3874.41,6,3.0,3,21.98,100.00
7,105306,Spain,low_value,social_media,4,9727.56,3960.80,4,0.0,4,20.41,100.00
8,105040,Germany,mid_value,social_media,6,9442.65,2523.90,6,0.0,5,28.64,100.00
9,105049,United Kingdom,mid_value,organic,10,9361.98,5360.42,10,2.0,2,21.82,100.00


Consulta SQL y resultado guardados correctamente.


In [11]:
# ============================================================
# 11. Resumen ejecutivo del análisis SQL
# SQL Business Analysis Project
# ============================================================

# Extraemos métricas principales
top_month_revenue = monthly_revenue.sort_values("total_net_revenue", ascending=False).iloc[0]
top_country_segment = customer_segments.sort_values("total_net_revenue", ascending=False).iloc[0]
top_product_category = product_performance.sort_values("total_net_revenue", ascending=False).iloc[0]
top_margin_category = product_performance.sort_values("gross_margin_pct", ascending=False).iloc[0]
top_fraud_method = fraud_risk.sort_values("fraud_rate_pct", ascending=False).iloc[0]
top_support_issue = support_sla.sort_values("tickets", ascending=False).iloc[0]

executive_summary = pd.DataFrame({
    "area": [
        "Business performance",
        "Monthly revenue",
        "Customer segments",
        "Product performance",
        "Product margin",
        "Fraud risk",
        "Support operations"
    ],
    "finding": [
        f"The business generated {business_kpis.loc[0, 'total_net_revenue']:,.2f} in net revenue and {business_kpis.loc[0, 'total_gross_profit']:,.2f} in gross profit.",
        f"The strongest revenue month was {top_month_revenue['order_month']} with {top_month_revenue['total_net_revenue']:,.2f} in net revenue.",
        f"The highest revenue customer segment was {top_country_segment['customer_segment']} in {top_country_segment['country']}.",
        f"The top product category by revenue was {top_product_category['category']} with {top_product_category['total_net_revenue']:,.2f}.",
        f"The highest margin category was {top_margin_category['category']} with a gross margin of {top_margin_category['gross_margin_pct']}%.",
        f"The payment method with the highest fraud rate was {top_fraud_method['payment_method']} at {top_fraud_method['fraud_rate_pct']}%.",
        f"The most frequent support issue was {top_support_issue['ticket_category']} with priority {top_support_issue['priority']}."
    ]
})

business_recommendations = pd.DataFrame({
    "priority": [
        "High",
        "High",
        "High",
        "Medium",
        "Medium",
        "Medium"
    ],
    "recommendation": [
        "Prioritize revenue growth actions in the strongest customer segments and countries.",
        "Monitor payment methods with higher fraud and chargeback rates.",
        "Review low-margin product categories and discount strategy.",
        "Use monthly revenue trends to detect seasonality and demand changes.",
        "Connect customer support issues with customer value and revenue impact.",
        "Build recurring SQL reports for management and operational teams."
    ]
})

# Guardar resultados
executive_summary.to_csv(
    project_root / "reports" / "executive_summary.csv",
    index=False
)

business_recommendations.to_csv(
    project_root / "reports" / "business_recommendations.csv",
    index=False
)

print("Resumen ejecutivo:")
display(executive_summary)

print("Recomendaciones de negocio:")
display(business_recommendations)

print("Archivos guardados correctamente en:")
print(project_root / "reports")

Resumen ejecutivo:


,area,finding
0,Business performance,"The business generated 8,925,618.18 in net rev..."
1,Monthly revenue,The strongest revenue month was 2024-12 with 7...
2,Customer segments,The highest revenue customer segment was low_v...
3,Product performance,The top product category by revenue was electr...
4,Product margin,The highest margin category was electronics wi...
5,Fraud risk,The payment method with the highest fraud rate...
6,Support operations,The most frequent support issue was account_ac...


Recomendaciones de negocio:


,priority,recommendation
0,High,Prioritize revenue growth actions in the stron...
1,High,Monitor payment methods with higher fraud and ...
2,High,Review low-margin product categories and disco...
3,Medium,Use monthly revenue trends to detect seasonali...
4,Medium,Connect customer support issues with customer ...
5,Medium,Build recurring SQL reports for management and...


Archivos guardados correctamente en:
/content/sql-business-analysis-project/reports


In [12]:
# ============================================================
# 12. Archivos preparados para dashboard
# SQL Business Analysis Project
# ============================================================

dashboard_path = project_root / "dashboard"
dashboard_path.mkdir(parents=True, exist_ok=True)

dashboard_files = {
    "dashboard_business_kpis.csv": business_kpis,
    "dashboard_monthly_revenue.csv": monthly_revenue,
    "dashboard_customer_segments.csv": customer_segments,
    "dashboard_product_performance.csv": product_performance,
    "dashboard_fraud_risk.csv": fraud_risk,
    "dashboard_support_sla.csv": support_sla,
    "dashboard_customer_360.csv": customer_360,
    "dashboard_executive_summary.csv": executive_summary,
    "dashboard_business_recommendations.csv": business_recommendations
}

for filename, dataframe in dashboard_files.items():
    dataframe.to_csv(dashboard_path / filename, index=False)

dashboard_summary = pd.DataFrame({
    "file": list(dashboard_files.keys()),
    "description": [
        "General business KPIs",
        "Monthly revenue and margin analysis",
        "Customer analysis by country and segment",
        "Product category performance",
        "Fraud and chargeback risk by payment method",
        "Support SLA analysis",
        "Customer 360 view combining revenue, fraud and support",
        "Executive SQL analysis summary",
        "Business recommendations"
    ]
})

display(dashboard_summary)

print("Archivos preparados para dashboard correctamente en:")
print(dashboard_path)

,file,description
0,dashboard_business_kpis.csv,General business KPIs
1,dashboard_monthly_revenue.csv,Monthly revenue and margin analysis
2,dashboard_customer_segments.csv,Customer analysis by country and segment
3,dashboard_product_performance.csv,Product category performance
4,dashboard_fraud_risk.csv,Fraud and chargeback risk by payment method
5,dashboard_support_sla.csv,Support SLA analysis
6,dashboard_customer_360.csv,"Customer 360 view combining revenue, fraud and..."
7,dashboard_executive_summary.csv,Executive SQL analysis summary
8,dashboard_business_recommendations.csv,Business recommendations


Archivos preparados para dashboard correctamente en:
/content/sql-business-analysis-project/dashboard


In [13]:
# ============================================================
# 13. README, requirements y estructura final del proyecto
# SQL Business Analysis Project
# ============================================================

readme_content = """
# SQL Business Analysis Project

## Project objective

This project demonstrates SQL-based business analysis using a simulated e-commerce and operations dataset.

The objective is to analyze revenue, profitability, customers, products, payment risk, fraud indicators and support operations using SQL queries.

This project is focused on practical SQL skills for Data Analyst, BI Analyst, Fraud Analyst, Risk Analyst and Operations Analyst roles.

---

## Dataset

The project uses simulated business tables:

- customers
- products
- orders
- transactions
- support_tickets

---

## Analysis included

- General business KPIs
- Monthly revenue and margin analysis
- Customer analysis by country and segment
- Product category performance
- Fraud and chargeback risk by payment method
- Support SLA analysis
- Customer 360 view combining revenue, fraud and support
- Executive summary and business recommendations
- Dashboard-ready datasets

---

## Main KPIs

- Total orders: 25,000
- Active customers: 7,627
- Total net revenue: 8,925,618.18
- Total gross profit: 4,272,047.86
- Average order value: 357.02
- Average profit per order: 170.88
- Gross margin: 47.86%

---

## Key findings

1. The business generated 8,925,618.18 in net revenue.
2. Total gross profit reached 4,272,047.86.
3. The strongest revenue month was 2024-12.
4. Electronics was the top product category by revenue.
5. Electronics also showed the highest margin.
6. Card payments showed the highest fraud rate.
7. Paypal was the second highest-risk payment method.
8. Customer 360 analysis connects revenue, fraud and support data.
9. SQL joins were used across customers, orders, products, transactions and tickets.
10. The project produces reusable CSV outputs for reporting and dashboards.

---

## Business recommendations

- Prioritize growth actions in the strongest customer segments and countries.
- Monitor payment methods with higher fraud and chargeback rates.
- Review low-margin product categories and discount strategy.
- Use monthly revenue trends to detect seasonality and demand changes.
- Connect support issues with customer value and revenue impact.
- Build recurring SQL reports for management and operational teams.

---

## SQL files

The `sql/` folder includes:

- 01_business_kpis.sql
- 02_monthly_revenue.sql
- 03_customer_segments.sql
- 04_product_performance.sql
- 05_fraud_risk.sql
- 06_support_sla.sql
- 07_customer_360.sql

---

## Tools used

- SQL
- DuckDB
- Python
- Pandas
- NumPy
- Google Colab
- CSV datasets

---

## Repository structure

sql-business-analysis-project/
- data/raw/
- data/processed/
- dashboard/
- reports/
- sql/
- 01_sql_business_analysis_project.ipynb
- README.md
- requirements.txt

---

## Author

Project created by Juan Carlos Garcia Balmaseda as part of a professional Data Analyst / BI Analyst / Risk Analyst portfolio.
"""

requirements_content = """
pandas
numpy
duckdb
"""

readme_path = project_root / "README.md"
requirements_path = project_root / "requirements.txt"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content.strip())

with open(requirements_path, "w", encoding="utf-8") as f:
    f.write(requirements_content.strip())

print("README.md creado correctamente en:")
print(readme_path)

print("\nrequirements.txt creado correctamente en:")
print(requirements_path)

print("\nContenido principal del proyecto:")
for item in sorted(project_root.iterdir()):
    print("-", item.name)

README.md creado correctamente en:
/content/sql-business-analysis-project/README.md

requirements.txt creado correctamente en:
/content/sql-business-analysis-project/requirements.txt

Contenido principal del proyecto:
- README.md
- dashboard
- data
- reports
- requirements.txt
- sql


In [14]:
# ============================================================
# 14. Crear ZIP final del proyecto
# SQL Business Analysis Project
# ============================================================

import shutil
from pathlib import Path
from google.colab import files

project_folder = Path("/content/sql-business-analysis-project")
zip_file = Path("/content/sql-business-analysis-project.zip")

print("¿Existe carpeta proyecto?:", project_folder.exists())

if zip_file.exists():
    zip_file.unlink()

shutil.make_archive(
    base_name="/content/sql-business-analysis-project",
    format="zip",
    root_dir="/content",
    base_dir="sql-business-analysis-project"
)

print("ZIP creado correctamente en:")
print(zip_file)

files.download(str(zip_file))

¿Existe carpeta proyecto?: True
ZIP creado correctamente en:
/content/sql-business-analysis-project.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>